# 12. Clean End-to-End OCR Benchmark

10개 카드의 PDF → OCR → 정규화 → JSON 구조화 → 평가를 한 실행 단위로 묶기 위한 안전한 시작 골격이다. 기존 01~11 결과는 변경하거나 복사하지 않고 baseline 후보로만 참조한다.

`12_clean_end_to_end_runner.py`는 OCR·strict 구조화 stage와 공통 run lock/cache/status를 구현하고, `12_clean_end_to_end_evaluator.py`는 coverage-aware TXT·structured·critical v2 오프라인 평가를 JSON/CSV로 기록한다. 이 노트북은 계약과 preflight만 실행하며 provider 호출 셀은 두지 않는다.


In [ ]:
import hashlib
import json
import os
import re
import tempfile
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import fitz


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'data/ocr_benchmark/gold/structured').is_dir():
            return candidate.resolve()
    raise RuntimeError('프로젝트 루트를 찾지 못했습니다.')


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / 'notebooks/data/12_clean_end_to_end'
CURRENT_ROOT = DATA_ROOT / 'current'
RUN_STAGES = ('raw', 'normalized', 'structured', 'evaluated', 'status')
CRITICAL_RULES_PATH = PROJECT_ROOT / 'data/ocr_benchmark/gold/critical_rules/critical_rules_v2.json'
EXPECTED_DOCUMENTS = 10
EXPECTED_PAGES = 50
MAX_DOCUMENTS = 10
MAX_PAGES = 50
MAX_CALLS = 140
MAX_STRUCTURE_CALLS = 60
MAX_ATTEMPTS = 1

LIVE_API = False
RUN_FLAGS = {
    'openai_luna_ocr': False,
    'openai_terra_ocr': False,
    'upstage_ocr': False,
    'field_extraction': False,
}


In [ ]:
OPENAI_OCR_MODELS = ('gpt-5.6-luna', 'gpt-5.6-terra')
OPENAI_IMAGE_DETAIL = 'original'
FIELD_EXTRACTION_MODEL = 'gpt-5.6-luna'
OCR_ENGINES = {
    'openai_luna': {'provider': 'openai', 'model': OPENAI_OCR_MODELS[0], 'detail': OPENAI_IMAGE_DETAIL},
    'openai_terra': {'provider': 'openai', 'model': OPENAI_OCR_MODELS[1], 'detail': OPENAI_IMAGE_DETAIL},
    'upstage': {'provider': 'upstage', 'model': os.getenv('UPSTAGE_OCR_MODEL', 'document-parse')},
}
UPSTAGE_REQUEST_CONFIG = {
    'model': 'document-parse',
    'ocr': 'force',
    'coordinates': True,
    'output_formats': ['html', 'markdown'],
}

OCR_PROMPT = '''카드 상품설명서 이미지를 원문 순서대로 정확히 전사하세요.
숫자, 단위, 부호, 적용 대상, 제외 조건, 전월 실적 조건과 표의 행·열 관계를 보존하세요.
보이지 않는 내용을 추측하거나 요약하지 말고, 읽을 수 없는 부분은 [ILLEGIBLE]로 표시하세요.'''.strip()

STRUCTURING_PROMPT = '''OCR 전사에서 카드 혜택·수수료 필드를 추출하세요.
제공된 ID와 JSON shape만 사용하고, OCR에 없는 값은 추측하지 말고 null로 반환하세요.
모든 엔진에 같은 기준을 적용하세요. 숫자는 surface_text, normalized_value, unit을 분리하고
normalized_value와 unit은 가능한 경우 KRW, RATIO, COUNT_PER_DAY 같은 명시적 단위로 정규화하세요.
표는 원문의 열 순서와 행 순서를 그대로 보존하세요.'''.strip()


In [ ]:
CARD_SPECS = (
    ('BC', 'BC_Biz_AirMoney'),
    ('NH', 'NH_Namu_NH'),
    ('hana', 'Hana_One_More_SOHO'),
    ('hyundai', 'Hyundai_The_Orange_20260330'),
    ('ibk', 'IBK_Point3.8(Credit)'),
    ('kookmin', 'Kookmin_Friend_20210917'),
    ('lotte', 'Lotte_LOCA_LIKIT_Eat'),
    ('samsung', 'Samsung_iD_ALL'),
    ('shinhan', 'Shinhan_Toss_Mr.Life_20251231'),
    ('woori', 'Woori_Classic_EVERY_MILE_SKYPASS'),
)

COVERAGE_POLICY = {
    f'{issuer}/{card_name}': (
        {'annotation_scope': 'selected_excerpt', 'full_page_cer': 'excluded', 'reason': 'structured gold가 selected excerpts라고 명시함'}
        if (issuer, card_name) == ('BC', 'BC_Biz_AirMoney')
        else {'annotation_scope': 'incomplete_or_ambiguous', 'full_page_cer': 'excluded', 'reason': 'gold raw가 불완전하거나 범위가 모호함'}
        if (issuer, card_name) == ('ibk', 'IBK_Point3.8(Credit)')
        else {
            'annotation_scope': 'full_page_candidate',
            'full_page_cer': 'excluded_until_visual_audit',
            'reason': '시각 감사 전 후보이며 full-page로 확정하지 않음',
            **({'audit_note': '특히 PDF 1~2쪽 시각 확인 권장'} if issuer == 'hyundai' else {}),
            **({'parser_note': '[page1]과 [page 1] 모두 허용; marker 리터럴을 강제하지 않음'} if issuer == 'woori' else {}),
        }
    )
    for issuer, card_name in CARD_SPECS
}


def card_inputs(issuer: str, card_name: str) -> dict[str, Path]:
    return {
        'pdf': PROJECT_ROOT / 'data/raw' / issuer / f'{card_name}.pdf',
        'gold_raw': PROJECT_ROOT / 'data/ocr_benchmark/gold/raw' / issuer / f'{card_name}.txt',
        'gold_structured': PROJECT_ROOT / 'data/ocr_benchmark/gold/structured' / issuer / f'{card_name}.json',
    }


def preflight() -> dict[str, Any]:
    if len(CARD_SPECS) != EXPECTED_DOCUMENTS or len(set(CARD_SPECS)) != EXPECTED_DOCUMENTS:
        raise ValueError(f'CARD_SPECS는 중복 없는 {EXPECTED_DOCUMENTS}개 카드여야 합니다.')

    cards = []
    missing = [CRITICAL_RULES_PATH] if not CRITICAL_RULES_PATH.is_file() else []
    for issuer, card_name in CARD_SPECS:
        paths = card_inputs(issuer, card_name)
        missing.extend(path for path in paths.values() if not path.is_file())
        page_count = None
        if paths['pdf'].is_file():
            with paths['pdf'].open('rb') as pdf_handle:
                if pdf_handle.read(5) != b'%PDF-':
                    raise ValueError(f'PDF 헤더가 아님: {paths["pdf"]}')
            with fitz.open(paths['pdf']) as document:
                page_count = len(document)
        if paths['gold_raw'].is_file() and not paths['gold_raw'].read_text(encoding='utf-8').strip():
            raise ValueError(f'빈 gold raw TXT: {paths["gold_raw"]}')
        if paths['gold_structured'].is_file():
            value = json.loads(paths['gold_structured'].read_text(encoding='utf-8'))
            if not isinstance(value, dict):
                raise ValueError(f'gold structured JSON 객체가 아님: {paths["gold_structured"]}')
        cards.append({'issuer': issuer, 'card_name': card_name, 'page_count': page_count, 'inputs': paths})

    if missing:
        joined = '\n'.join(str(path) for path in missing)
        raise FileNotFoundError(f'필수 입력 파일 누락:\n{joined}')
    critical_rules = json.loads(CRITICAL_RULES_PATH.read_text(encoding='utf-8'))
    if not isinstance(critical_rules, dict):
        raise ValueError(f'critical rules JSON 객체가 아님: {CRITICAL_RULES_PATH}')
    total_pages = sum(card['page_count'] for card in cards)
    if total_pages != EXPECTED_PAGES:
        raise ValueError(f'PDF 총 페이지가 {EXPECTED_PAGES}가 아님: {total_pages}')
    return {
        'card_count': len(cards),
        'page_count': total_pages,
        'counts': {kind: sum(card['inputs'][kind].is_file() for card in cards) for kind in ('pdf', 'gold_raw', 'gold_structured')},
        'critical_rules': CRITICAL_RULES_PATH,
        'cards': cards,
    }


def dry_run_plan(checked: dict[str, Any] | None = None) -> dict[str, Any]:
    checked = checked or preflight()
    calls = {
        'openai_ocr': checked['page_count'] * len(OPENAI_OCR_MODELS),
        'upstage_ocr': checked['card_count'],
        'field_extraction': checked['card_count'] * (len(OPENAI_OCR_MODELS) + 1),
    }
    plan = {
        'documents': checked['card_count'],
        'pages': checked['page_count'],
        'calls': calls,
        'ocr_total_calls': calls['openai_ocr'] + calls['upstage_ocr'],
        'total_calls': sum(calls.values()),
        'attempts_per_call': MAX_ATTEMPTS,
        'oracle_included': False,
    }
    if plan['documents'] > MAX_DOCUMENTS or plan['pages'] > MAX_PAGES or plan['total_calls'] > MAX_CALLS:
        raise RuntimeError(f'dry-run plan이 호출 상한을 초과함: {plan}')
    return plan


In [ ]:
OPTIONAL_BASELINE_CANDIDATES = (
    PROJECT_ROOT / 'notebooks/data/09_core_numeric_condition_ocr_evaluation/api_original_repeatability_summary.json',
    PROJECT_ROOT / 'notebooks/data/09_core_numeric_condition_ocr_evaluation/runs/20260807T144000Z_upstage_baseline/summary.json',
    PROJECT_ROOT / 'notebooks/data/10_relational_critical_fact_evaluation/runs/20260808T082345Z/summary.json',
    PROJECT_ROOT / 'notebooks/data/11_numeric_error_attribution/runs/20260810T060240Z/summary.json',
    *(PROJECT_ROOT / root / issuer / f'{card_name}{suffix}'
      for root, suffix in (
          ('data/ocr_benchmark/upstage_raw', '.json'),
          ('data/ocr_benchmark/text/upstage', '.md'),
          ('data/ocr_benchmark/normalized/upstage', '.json'),
      )
      for issuer, card_name in CARD_SPECS),
)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def file_reference(path: Path) -> dict[str, Any]:
    relative = path.resolve().relative_to(PROJECT_ROOT)
    return {'path': relative.as_posix(), 'sha256': sha256_file(path), 'bytes': path.stat().st_size}


def optional_file_reference(path: Path) -> dict[str, Any]:
    relative = path.resolve().relative_to(PROJECT_ROOT).as_posix()
    if not path.is_file():
        return {'path': relative, 'available': False}
    return {'path': relative, 'available': True, 'sha256': sha256_file(path), 'bytes': path.stat().st_size}


def atomic_write_json(path: Path, value: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = None
    try:
        with tempfile.NamedTemporaryFile('w', encoding='utf-8', dir=path.parent, prefix=f'.{path.name}.', suffix='.tmp', delete=False) as handle:
            temporary = Path(handle.name)
            json.dump(value, handle, ensure_ascii=False, indent=2)
            handle.write('\n')
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary, path)
    except Exception:
        if temporary is not None:
            temporary.unlink(missing_ok=True)
        raise


def failed_item_status(item_id: str, stage: str, error: Exception, attempt: int, fingerprint: str | None = None) -> dict[str, Any]:
    if not 1 <= attempt <= MAX_ATTEMPTS:
        raise ValueError(f'attempt는 1..{MAX_ATTEMPTS} 범위여야 합니다.')
    return {
        'item_id': item_id,
        'stage': stage,
        'status': 'failed',
        'attempt': attempt,
        'request_fingerprint': fingerprint,
        'error_type': type(error).__name__,
        'error_message': str(error),
        'failed_at': datetime.now(timezone.utc).isoformat(),
    }


def record_item_failure(path: Path, item_id: str, stage: str, error: Exception, attempt: int, fingerprint: str | None = None) -> dict[str, Any]:
    status = failed_item_status(item_id, stage, error, attempt, fingerprint)
    atomic_write_json(path, status)
    return status


def build_run_manifest(run_id: str, checked: dict[str, Any] | None = None) -> dict[str, Any]:
    checked = checked or preflight()
    inputs = []
    for card in checked['cards']:
        inputs.append({
            'issuer': card['issuer'],
            'card_name': card['card_name'],
            **{kind: file_reference(path) for kind, path in card['inputs'].items()},
        })
    return {
        'schema_version': 'clean_end_to_end_run_v2',
        'run_id': run_id,
        'created_at': datetime.now(timezone.utc).isoformat(),
        'required_inputs': {
            'cards': inputs,
            'critical_rules': file_reference(checked['critical_rules']),
        },
        'optional_references': [optional_file_reference(path) for path in OPTIONAL_BASELINE_CANDIDATES],
        'dry_run_plan': dry_run_plan(checked),
        'coverage_policy': COVERAGE_POLICY,
        'settings': {
            'live_api': LIVE_API,
            'run_flags': dict(RUN_FLAGS),
            'ocr_engines': OCR_ENGINES,
            'upstage_request_config': UPSTAGE_REQUEST_CONFIG,
            'field_extraction_model': FIELD_EXTRACTION_MODEL,
            'prompts': {'ocr': OCR_PROMPT, 'structuring': STRUCTURING_PROMPT},
            'limits': {
                'max_documents': MAX_DOCUMENTS,
                'max_pages': MAX_PAGES,
                'max_calls': MAX_CALLS,
                'max_structure_calls': MAX_STRUCTURE_CALLS,
                'max_attempts': MAX_ATTEMPTS,
            },
        },
    }


def create_run(run_id: str | None = None) -> Path:
    run_id = run_id or datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    if not re.fullmatch(r'\d{8}T\d{6}Z', run_id):
        raise ValueError('run_id 형식은 YYYYMMDDTHHMMSSZ여야 합니다.')
    run_root = CURRENT_ROOT / 'runs' / run_id
    if run_root.exists():
        raise FileExistsError(f'기존 실행을 덮어쓰지 않습니다: {run_root}')

    checked = preflight()
    for stage in RUN_STAGES:
        (run_root / stage).mkdir(parents=True, exist_ok=False)
    manifest = build_run_manifest(run_id, checked)
    atomic_write_json(run_root / 'run_manifest.json', manifest)
    return run_root


In [ ]:
NORMALIZED_V2_SCHEMA = {
    'schema_version': 'normalized_ocr_v2',
    'canonical_text': 'pages[].text',
    'required_page_fields': ['page_num', 'text', 'marker_collisions'],
    'coordinates': 'optional provider evidence; omitted for OpenAI',
}
PAGE_TXT_MARKER = re.compile(r'^\[PAGE\s*(\d+)\]$', re.IGNORECASE | re.MULTILINE)
PAGE_COLLISION_PATTERN = re.compile(r'^\[PAGE\s*\d+\]\s*$', re.IGNORECASE | re.MULTILINE)
PANEL_MARKER_PATTERN = re.compile(r'^\[(?:left|center|right)_panel\]\s*$', re.IGNORECASE | re.MULTILINE)


def canonical_text(text: str) -> str:
    if not isinstance(text, str):
        raise TypeError('pages[].text는 문자열이어야 합니다.')
    return text.replace('\r\n', '\n').replace('\r', '\n').strip()


def marker_collisions(text: str) -> dict[str, int]:
    collisions = {
        'page_marker': len(PAGE_COLLISION_PATTERN.findall(text)),
        'panel_marker': len(PANEL_MARKER_PATTERN.findall(text)),
    }
    return {name: count for name, count in collisions.items() if count}


def validate_normalized_v2(document: dict[str, Any]) -> dict[str, Any]:
    if document.get('schema_version') != 'normalized_ocr_v2':
        raise ValueError('normalized schema_version이 normalized_ocr_v2가 아닙니다.')
    pages = document.get('pages')
    if not isinstance(pages, list) or not pages:
        raise ValueError('pages는 비어 있지 않은 목록이어야 합니다.')
    page_numbers = [page.get('page_num') for page in pages]
    if page_numbers != list(range(1, len(pages) + 1)):
        raise ValueError(f'page_num은 1부터 연속이어야 합니다: {page_numbers}')
    for page in pages:
        if page.get('text') != canonical_text(page.get('text')):
            raise ValueError(f'pages[{page["page_num"]}].text가 canonical text가 아닙니다.')
        if page.get('marker_collisions') != marker_collisions(page['text']):
            raise ValueError(f'pages[{page["page_num"]}] marker collision 기록이 일치하지 않습니다.')
        if str(document.get('provider', '')).lower() == 'openai' and 'coordinates' in page:
            raise ValueError('OpenAI normalized page에는 좌표를 생성하지 않습니다.')
    return document


def build_normalized_v2(provider: str, model: str, source_pdf: str, pages: list[dict[str, Any]], fingerprint: str) -> dict[str, Any]:
    normalized_pages = []
    for page in pages:
        text = canonical_text(page.get('text'))
        normalized_page = {
            'page_num': page.get('page_num'),
            'text': text,
            'marker_collisions': marker_collisions(text),
        }
        if 'coordinates' in page:
            if provider.lower() == 'openai':
                raise ValueError('OpenAI 응답에 가짜 좌표를 추가할 수 없습니다.')
            normalized_page['coordinates'] = page['coordinates']
        normalized_pages.append(normalized_page)
    document = {
        'schema_version': 'normalized_ocr_v2',
        'provider': provider,
        'model': model,
        'source_pdf': source_pdf,
        'request_fingerprint': fingerprint,
        'pages': normalized_pages,
    }
    return validate_normalized_v2(document)


def normalized_to_txt(document: dict[str, Any]) -> str:
    validate_normalized_v2(document)
    if any(page['marker_collisions'].get('page_marker') for page in document['pages']):
        raise ValueError('provider text의 PAGE marker 충돌로 TXT를 안전하게 직렬화할 수 없습니다.')
    return '\n\n'.join(f'[PAGE {page["page_num"]}]\n{page["text"]}' for page in document['pages']) + '\n'


def parse_normalized_txt(text: str) -> list[dict[str, Any]]:
    matches = list(PAGE_TXT_MARKER.finditer(text))
    if not matches or text[:matches[0].start()].strip():
        raise ValueError('TXT는 orchestration PAGE marker로 시작해야 합니다.')
    pages = []
    for index, match in enumerate(matches):
        end = matches[index + 1].start() if index + 1 < len(matches) else len(text)
        page_text = canonical_text(text[match.end():end])
        pages.append({
            'page_num': int(match.group(1)),
            'text': page_text,
            'marker_collisions': marker_collisions(page_text),
        })
    if [page['page_num'] for page in pages] != list(range(1, len(pages) + 1)):
        raise ValueError('TXT PAGE marker는 1부터 연속이어야 합니다.')
    return pages


def gold_text_for_evaluation(text: str) -> dict[str, Any]:
    panel_marker_count = len(PANEL_MARKER_PATTERN.findall(text))
    return {
        'text': PANEL_MARKER_PATTERN.sub('', text),
        'removed_panel_markers': panel_marker_count,
    }


In [ ]:
def canonical_json_bytes(value: Any) -> bytes:
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',', ':')).encode('utf-8')


def sha256_value(value: Any) -> str:
    payload = value.encode('utf-8') if isinstance(value, str) else canonical_json_bytes(value)
    return hashlib.sha256(payload).hexdigest()


def build_request_fingerprint(provider: str, model: str, config: dict[str, Any], prompt: str, source_path: Path, source_kind: str, schema: Any) -> dict[str, Any]:
    if source_kind not in {'pdf', 'image'}:
        raise ValueError('source_kind는 pdf 또는 image여야 합니다.')
    source = file_reference(source_path)
    components = {
        'provider': provider,
        'model': model,
        'config': config,
        'prompt_sha256': sha256_value(prompt),
        'source': {'kind': source_kind, 'path': source['path'], 'sha256': source['sha256']},
        'schema_sha256': sha256_value(schema),
    }
    return {'fingerprint': sha256_value(components), 'components': components}


def validate_cache_entry(path: Path, expected_fingerprint: str | dict[str, Any]) -> dict[str, Any]:
    expected = expected_fingerprint['fingerprint'] if isinstance(expected_fingerprint, dict) else expected_fingerprint
    if not path.is_file():
        return {'hit': False, 'reason': 'missing'}
    try:
        entry = json.loads(path.read_text(encoding='utf-8'))
    except (OSError, json.JSONDecodeError):
        return {'hit': False, 'reason': 'invalid_json'}
    if not isinstance(entry, dict):
        return {'hit': False, 'reason': 'invalid_envelope'}
    if entry.get('status') != 'succeeded':
        return {'hit': False, 'reason': 'incomplete_status'}
    if entry.get('request_fingerprint') != expected:
        return {'hit': False, 'reason': 'fingerprint_mismatch'}
    if 'payload' not in entry:
        return {'hit': False, 'reason': 'missing_payload'}
    return {'hit': True, 'reason': 'validated', 'entry': entry}


In [ ]:
def require_live_api(flag: str) -> None:
    if flag not in RUN_FLAGS:
        raise KeyError(f'알 수 없는 실행 플래그: {flag}')
    if not LIVE_API:
        raise RuntimeError('LIVE_API=False: 유료 API 호출이 차단되었습니다.')
    if not RUN_FLAGS[flag]:
        raise RuntimeError(f'{flag}=False: 해당 단계 호출이 차단되었습니다.')


# 노트북은 provider를 호출하지 않는다. 실제 OCR/구조화는 runner의 명시적
# --stage, --engines, --live-api guard와 per-run lock을 통과해야 한다.


## 다음 단계 계약

1. 실행 승인 후에만 runner에 `--live-api`, 명시적 `--stage`, `--engines`를 함께 준다. Upstage는 OpenAI 모델 변형이 아닌 독립 OCR 엔진이다.
2. provider 호출 직전에 `require_live_api()`를 통과시킨다. 정답 TXT·JSON은 OCR/구조화 프롬프트에 전달하지 않는다.
3. provider 원문 응답은 `raw/`, canonical `pages[].text`는 `normalized/`, 필드 JSON은 `structured/`, coverage policy를 통과한 gold 대조 결과는 `evaluated/`에 저장한다.
4. 호출 전 예상 페이지 수·비용·대상 엔진을 확인하고, 10개 카드 전체 실행 전 카드 1개 smoke run을 별도 `run_id`로 검증한다.
5. 기존 09~11 및 Upstage 후보는 optional reference로만 비교하며 누락을 clean run 실패로 처리하지 않는다.
6. BC selected excerpt는 structured gold label의 같은 page에서 raw_start_marker 줄을 NFKC·공백 정규화하고 선행 Markdown heading만 제거해 exact 비교한다. code fence 구분 줄 외에는 제거하지 않으며 marker 누락·다른 페이지 marker는 unavailable로 둔다. IBK incomplete/ambiguous는 공식 full-page 집계에서 제외한다. 나머지 8개 `full_page_candidate`만 시각 감사 전 candidate aggregate로 별도 표시한다.
7. 구조화는 30개 normalized 각각을 상수 Luna와 동일 value-less strict contract로 처리한다. gold ID/page/type·shape/table 열 수만 허용하고 value/context terms는 제외한다. 정상 계획은 30회, 전역 최대 60회이며 context-limit 오류 때만 page 순서로 분할한다.
8. `--stage evaluate --execute-offline`은 외부 호출 없이 generation이 일치하는 JSON/CSV를 만든다. 구조화 결과가 없으면 exact 지표 null과 incomplete nonzero이며, 검증 시에만 `--allow-incomplete`를 명시한다. critical table-row는 explicit table/row/column locator가 없으면 pass 분모에서 제외하고 heuristic은 진단으로만 남긴다. unit accuracy는 relation-scorable numeric source의 expected-unit 계약 전체를 분모로 삼아 prediction unit 누락도 atomic relation과 unit accuracy의 오답으로 처리한다. field source canonical type은 unit을 별도로 요구하지 않으며 TXT preview·structured·critical 결과는 통합값과 엔진별 값을 함께 기록한다.
9. live/평가 실행은 쓰기 전에 per-run lock을 획득한다. stale lock은 같은 host의 기록 PID가 종료됐을 때만 명시적 `--recover-stale-lock`로 복구한다.
